# Train input 50차원 변환 과정 (단계별)

예: `1-Aminonaphthalene` + `Temperature=200°C` 가 GP 학습 입력(`x_train` 한 행)으로 바뀌는 과정을
`test_20260530.yaml` + `topo_physchem` 설정과 동일한 파이프라인으로 따라갑니다.

**전제:** 먼저 학습 CSV로 `condition_transformer` / `gp_input_scaler`를 fit 해야 합니다.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.features.topo_physchem import FEATURE_NAMES, encode_molecule

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "test_20260530.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data_extended.csv"

REACTANT_RAW = "1-Aminonaphthalene"
TEMPERATURE_C = 200.0

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
artifacts = pipeline.train_from_csv(DATA_PATH)

print("featuriser:", config.featurization.featuriser)
print("standardize_gp_inputs:", config.optimization.standardize_gp_inputs)
print("x_train shape:", artifacts.x_train.shape)

In [ ]:
def explain_input_transform(
    pipeline: ProphetGPPipeline,
    artifacts,
    reactant_raw: str,
    temperature_c: float,
    *,
    verbose: bool = True,
) -> dict:
    """학습 파이프라인과 동일한 단계로 50차원 입력을 만들고 중간 결과를 반환한다."""
    delim = pipeline.config.data.reactant_delimiter
    featuriser = pipeline.config.featurization.featuriser

    # --- Step 1: 원본 입력 ---
    tokens = [t.strip() for t in reactant_raw.split(delim) if t.strip()]

    # --- Step 2: 이름/CAS → canonical SMILES ---
    smiles_list = [
        pipeline.dataset_service.resolver.to_canonical_smiles(tok) for tok in tokens
    ]

    # --- Step 3: 분자별 topo_physchem (49차원) ---
    per_molecule = pipeline.featurizers.featurize(smiles_list, name=featuriser)
    mol_matrix = np.asarray(per_molecule, dtype=np.float32)
    if featuriser == "topo_physchem":
        mol_named = pd.DataFrame(mol_matrix, columns=FEATURE_NAMES)
    else:
        mol_named = pd.DataFrame(mol_matrix, columns=[f"mol_f{i}" for i in range(mol_matrix.shape[1])])

    # --- Step 4: 다중 반응물이면 분자 벡터 평균 pooling ---
    mol_vec = mol_matrix.mean(axis=0)

    # --- Step 5: Temperature → sklearn StandardScaler (학습 fit 기준) ---
    cond_cols = artifacts.condition_columns
    cond_df = pd.DataFrame([{cond_cols[0]: temperature_c}])
    x_cond = artifacts.condition_transformer.transform(cond_df)
    if hasattr(x_cond, "toarray"):
        x_cond = x_cond.toarray()
    cond_vec = np.asarray(x_cond, dtype=np.float32).reshape(-1)

    num_pipe = artifacts.condition_transformer.named_transformers_.get("numeric")
    temp_scaler = num_pipe.named_steps["scaler"] if num_pipe is not None else None
    temp_mean = float(temp_scaler.mean_[0]) if temp_scaler is not None else None
    temp_scale = float(temp_scaler.scale_[0]) if temp_scaler is not None else None

    # --- Step 6: 분자 + 조건 concat (GP min-max 이전, "물리" 결합 벡터) ---
    x_pre_gp = np.concatenate([mol_vec, cond_vec], axis=0).astype(np.float32)

    # --- Step 7: GP 입력 min-max [0,1] (config standardize_gp_inputs=True일 때) ---
    if artifacts.gp_input_scaler is not None:
        x_gp = artifacts.gp_input_scaler.transform(x_pre_gp.reshape(1, -1)).astype(np.float32).reshape(-1)
    else:
        x_gp = x_pre_gp.copy()

    # --- Step 8: 학습 행과 일치 확인 ---
    pred = pipeline.predict_targets(
        artifacts,
        {"reactants": reactant_raw, "Temperature": temperature_c},
    )
    x_via_api = pred.raw_features.reshape(-1)
    max_diff = float(np.max(np.abs(x_gp - x_via_api)))

    feature_labels = list(mol_named.columns) + [f"{cond_cols[0]}_standardized"]
    steps = {
        "1_raw_reactant": reactant_raw,
        "2_tokens": tokens,
        "3_canonical_smiles": smiles_list,
        "4_per_molecule_matrix": mol_matrix,
        "4_per_molecule_named": mol_named,
        "5_mol_vector_pooled": mol_vec,
        "6_temperature_c": temperature_c,
        "6_temperature_scaler_mean": temp_mean,
        "6_temperature_scaler_scale": temp_scale,
        "7_condition_vector": cond_vec,
        "8_x_pre_gp_scale": x_pre_gp,
        "9_x_gp_input": x_gp,
        "feature_labels": feature_labels,
        "max_diff_vs_predict_targets": max_diff,
    }

    if verbose:
        print("=" * 60)
        print("Step 1 | 원본 반응물 문자열")
        print(" ", reactant_raw)
        print("\nStep 2 | delimiter 분리 토큰")
        print(" ", tokens)
        print("\nStep 3 | canonical SMILES (RDKit/PubChem resolver)")
        for tok, smi in zip(tokens, smiles_list):
            print(f"   {tok!r} -> {smi}")
        print(f"\nStep 4 | featuriser='{featuriser}' per-molecule shape", mol_matrix.shape)
        display(mol_named)
        print("\nStep 5 | 반응물 벡터 (분자별 평균 pooling)")
        print(f"   dim={mol_vec.shape[0]}")
        display(pd.DataFrame({"feature": FEATURE_NAMES if featuriser == "topo_physchem" else mol_named.columns, "value": mol_vec}))
        print("\nStep 6 | Temperature 원값 (°C)")
        print(f"   T = {temperature_c}")
        if temp_scaler is not None:
            z = (temperature_c - temp_mean) / temp_scale
            print(f"   StandardScaler: (T - {temp_mean:.4f}) / {temp_scale:.4f} = {z:.6f}")
        print("\nStep 7 | 조건 벡터 (scaled)")
        print("  ", cond_vec)
        print("\nStep 8 | concat → x_pre_gp_scale (min-max 이전, dim=", x_pre_gp.shape[0], ")")
        display(pd.DataFrame({"idx": [f"x{i}" for i in range(len(x_pre_gp))], "name": feature_labels, "value": x_pre_gp}))
        if artifacts.gp_input_scaler is not None:
            print("\nStep 9 | MinMaxScaler → GP 입력 [0,1] (dim=", x_gp.shape[0], ")")
            display(pd.DataFrame({"idx": [f"x{i}" for i in range(len(x_gp))], "name": feature_labels, "value": x_gp}))
        else:
            print("\nStep 9 | GP 입력 = x_pre_gp (추가 스케일 없음)")
        print(f"\n검증 | predict_targets 와 max 차이: {max_diff:.2e}")

    return steps

steps = explain_input_transform(pipeline, artifacts, REACTANT_RAW, TEMPERATURE_C)

In [ ]:
# 학습 CSV에서 동일 (Mol.1, Temperature) 행과 x_train 직접 비교
df = pd.read_csv(DATA_PATH)
react_col = config.data.reactant_column
mask = (df[react_col].astype(str) == REACTANT_RAW) & (df["Temperature"] == TEMPERATURE_C)
print("CSV 행 (타깃 포함):")
display(df.loc[mask, [react_col, "Temperature", "Emission Peak", "FWHM"]])

row_idx = [i for i, r in enumerate(artifacts.reactant_inputs) if r == REACTANT_RAW]
print("\nartifacts 내 동일 반응물 행 인덱스:", row_idx)
for i in row_idx:
    t_raw = artifacts.training_query_inputs[i]["Temperature"]
    diff = np.max(np.abs(artifacts.x_train[i] - steps["9_x_gp_input"]))
    print(f"  i={i}  training T={t_raw}  |x_train - walkthrough| max={diff:.2e}")
    print(f"       y_train = {artifacts.y_train[i]}")

## 요약 (50차원 구성)

| 단계 | 차원 | 내용 |
|------|------|------|
| 3–5 | 49 | `topo_physchem` descriptor (단일 분자면 pooling 변화 없음) |
| 6–7 | 1 | `Temperature` → 학습 데이터 `StandardScaler` |
| 8 | 50 | 위 둘 concat (`x_pre_gp_scale`) |
| 9 | 50 | `standardize_gp_inputs=true` 이면 전체 min-max → **실제 GP `x_train`** |